In [1]:
from mofapy2.run.entry_point import entry_point
import pandas as pd
import numpy as np
import mofax as mfx

# initialise the entry point
ent = entry_point()


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        


In [3]:
# Preprocessed data loading
proteomics = pd.read_csv("C:/Users/49152/Downloads/Multi-omics/MOFA/input/5_MOFA_complied_preprocessing/proteomics_for_MOFA.csv")
transcriptomics = pd.read_csv("C:/Users/49152/Downloads/Multi-omics/MOFA/input/5_MOFA_complied_preprocessing/transcriptomics_for_MOFA.csv")

# Set Gene column as index
proteomics_mat = proteomics.set_index("Gene")
transcriptomics_mat = transcriptomics.set_index("Gene")

In [5]:
# Identify paried samples
prot_samples = pd.Index(proteomics_mat.columns)
rna_samples = pd.Index(transcriptomics_mat.columns)

common_samples = prot_samples.intersection(rna_samples)
prot_only = prot_samples.difference(rna_samples)
rna_only = rna_samples.difference(prot_samples)

print(f"Proteomics samples: {len(prot_samples)}")
print(f"Transcriptomics samples: {len(rna_samples)}")
print(f"Common paired samples: {len(common_samples)}")
print(f"Proteomics-only samples: {len(prot_only)}")
print(f"Transcriptomics-only samples: {len(rna_only)}")

if len(common_samples) == 0:
    raise ValueError("No shared SampleID values found between proteomics and transcriptomics matrices.")

# Sort common samples to ensure stable ordering
common_samples = common_samples.sort_values()

# Subset both views to paired samples only
proteomics_mat = proteomics_mat.loc[:, common_samples]
transcriptomics_mat = transcriptomics_mat.loc[:, common_samples]

# Final safety check
if not all(proteomics_mat.columns == transcriptomics_mat.columns):
    raise ValueError("Sample order mismatch between proteomics and transcriptomics after subsetting.")

print("Paired reference matrices created successfully.")
print(f"Proteomics matrix shape: {proteomics_mat.shape}")
print(f"Transcriptomics matrix shape: {transcriptomics_mat.shape}")

Proteomics samples: 68
Transcriptomics samples: 82
Common paired samples: 61
Proteomics-only samples: 7
Transcriptomics-only samples: 21
Paired reference matrices created successfully.
Proteomics matrix shape: (1500, 61)
Transcriptomics matrix shape: (1500, 61)


In [7]:
# Convert to long format
proteomics_long = (proteomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value").rename(columns={"Gene": "feature"}))
proteomics_long["view"] = "Proteomics"
proteomics_long["feature"] = proteomics_long["feature"].astype(str) + "_prot"

transcriptomics_long = (transcriptomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value").rename(columns={"Gene": "feature"}))
transcriptomics_long["view"] = "Transcriptomics"
transcriptomics_long["feature"] = transcriptomics_long["feature"].astype(str) + "_rna"

mofa_long = pd.concat([proteomics_long, transcriptomics_long], ignore_index=True)

print(mofa_long.head())
print(mofa_long["view"].value_counts())

        feature       sample     value        view
0   Eif2b4_prot  05J_C10_A10 -3.040062  Proteomics
1  Slc38a2_prot  05J_C10_A10       NaN  Proteomics
2     Wnk1_prot  05J_C10_A10  7.491947  Proteomics
3    Krt6a_prot  05J_C10_A10 -3.799535  Proteomics
4  Rabgef1_prot  05J_C10_A10  1.302259  Proteomics
view
Proteomics         91500
Transcriptomics    91500
Name: count, dtype: int64


In [9]:
ent.set_data_df(mofa_long, likelihoods = ["gaussian","gaussian"])


No "group" column found in the data frame, we will assume a common group for all samples...


Loaded group='single_group' view='Proteomics' with N=61 samples and D=1500 features...
Loaded group='single_group' view='Transcriptomics' with N=61 samples and D=1500 features...




In [11]:
ent.set_model_options(factors = 5)

Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (Proteomics): gaussian
- View 1 (Transcriptomics): gaussian




In [13]:
ent.set_train_options(convergence_mode = "slow", dropR2 = 0.001, gpu_mode = True, seed = 1)

In [ ]:
ent.build()
ent.run()

In [ ]:
# Save model
ent.save(outfile="C:/Users/49152/Downloads/Multi-omics/MOFA/output/single_cell_trained_model_MOFAcomplied.hdf5")